# NB04_fPINN — Robustness experiments (null test, history baselines, b_v ablation, encoder noise, more seeds)

**Purpose.** Fourth step of the fractional-PINN paper. It tests the interpretation of the results with new runs on the same plant, data and training protocol as NB02_fPINN / NB03_fPINN:

| Block | Question | What changes | Networks | Traj. | Seeds | α |
|---|---|---|---|---|---|---|
| `null` | Q1 | plant **without** fractional term (B = 0): does the fPINN invent B̂ ≠ 0? | PINN, HYBRID, fPINN, fHYBRID | 175 | 0–2 | – |
| `hist` | Q2 | integer baselines **with access to the history**: `MLPhist` (MLP fed the 39 L1 features D^αj q as extra inputs, 12 739 weights) and `PINNmem` (PINN + free linear memory Σ_j w_j D^αj q, 117 free gains instead of α, B) | MLPhist, PINNmem | 35, 175 | 0–4 | 0.5 |
| `main` | Q3 | more seeds of the main study (seeds 0–2 are read from the NB02 cache) | all 5 | 35, 175 | 0–4 (PINN, fPINN at 175: 0–9) | 0.5 |
| `sweep` | Q3 | more seeds of the α sweep (seed 0 from the NB02 cache) | PINN, HYBRID, fPINN, fHYBRID | 175 | 0–2 | 0.3, 0.7, 0.9 |
| `bv0` | Q4 | plant without the viscous term of τ_u (b_v = 0): does the +2…19 % bias of B̂ vanish? | PINN, HYBRID, fPINN, fHYBRID | 175 | 0–2 | 0.5 |
| `noise` | Q5 | encoder quantization 77 µrad (+0.5 count jitter), q̇, q̈ by Savitzky–Golay; labels are the true torque | PINN, HYBRID, fPINN, fHYBRID | 175 | 0–2 | 0.5 |

Closed loop (CTC + I of NB03, same gains, Adept 3 speeds × 0/6 kg) is then run for every new network of `main`, `sweep`, `hist` and `noise`; in `noise` the controller sees the quantized encoder through an α-β observer, and the nominal model and the oracle are run under the same measurement. Runs already in `NB03_fPINN_ClosedLoop.csv` are not repeated.

**Faster training, same networks.** NB02 spent most of its time in PyTorch's `DataLoader` assembling batches sample by sample. NB04 draws the **same shuffled batches** (it reproduces the random stream of `DataLoader(shuffle=True, generator=seed)` exactly) by indexing whole tensors, which is ~25× faster and leaves the training protocol of notebook 18 unchanged. A built-in check (`CHECK_LOADER`) trains one epoch both ways and confirms identical losses on your machine.

**One switch.** `QUICK = True` is a fast check of the whole pipeline (~2 min: one network per block at 35 trajectories and 2 epochs, closed loop at the middle speed). `QUICK = False` runs **everything above** in one go: 110 new networks and ~530 closed-loop runs, with nothing else to edit.

**Expected runtime (Ryzen 7 PRO 5850U, 16 GB): about 5 h, between 4 and 7 h** (≈ 4 h of training, measured here at 0.4–0.9 s per epoch with 175 trajectories and scaled ×2 for the reference laptop, plus ≈ 45 min of closed loop). The notebook prints its own estimate after every network from the speed it measures. Run it overnight. Every finished network is cached in `CACHE_DIR` with its weights in `WEIGHTS_OUT`, and every closed-loop run is appended to the CSV, so an interruption loses at most the network or run in progress: rerun and it resumes. Blocks run in priority order (null, hist, main, bv0, noise, sweep, then closed loop), so a partial run already holds the most important experiments. RAM ≈ 3 GB, printed after every network.

**Put in the same folder:** this notebook, the folder `NB02_fPINN_cache` and the `NB02_fPINN_*_e150.npz` files of NB02 (used to reuse seeds 0–2 and seed 0 of the sweep), and `NB03_fPINN_ClosedLoop.csv`. If they are missing, those runs are simply redone.

**Outputs:** `NB04_fPINN_TrainingLog.csv` (every network, NB02 rows included, with a `Scenario` column), `NB04_fPINN_ClosedLoop.csv` (NB03 rows included), `NB04_fPINN_Summary.csv`, `fig_nb04_fpinn.png`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/Anaconda: NumPy-SciPy (MKL) and PyTorch each ship an OpenMP runtime;
                                                        # without this the kernel can die silently ("OMP Error #15") on import
import gc, time, copy, random, pickle, numpy as np, pandas as pd
from math import gamma as Gamma
from scipy.signal import fftconvolve
import torch, torch.nn as nn                            # same import order as NB02 (numpy, scipy.signal, torch)
from torch.utils.data import TensorDataset, DataLoader
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt

# ---- settings ---------------------------------------------------------------------------------
QUICK        = True                # True: fast check (~2 min). False: every experiment of this notebook (see header).
B_FRAC       = 0.05                # [N m s^α] per joint (as NB02)
ALPHA_MAIN   = 0.5
TRAIN_CFG    = dict(batch_size=2048, lr=1e-3, weight_decay=1e-5, max_epochs=150,
                    patience=18, lr_patience=6, log_every=25)                      # notebook 18 / NB02
ALPHA_INIT   = 0.8
ENC_COUNTS   = 2048*4*10           # encoder lines x 4 (quadrature) x 10:1 gearbox
Q_ENC        = 2*np.pi/ENC_COUNTS  # 76.7 µrad
ENC_JITTER   = 0.5                 # Gaussian jitter before quantization [counts]
SG_WINDOW    = 81                  # Savitzky–Golay window at 2 kHz (40 ms), order 3, for q̇ and q̈ (identification data)
OBS_BW       = 50.0                # [Hz] bandwidth of the α-β observer that gives q, q̇ to the controller under encoder noise
CHECK_LOADER = True                # one epoch with DataLoader vs fast batches: losses must match
NB02_CACHE   = "NB02_fPINN_cache"  # reused rows (main seeds 0-2, sweep seed 0)
NB02_WEIGHTS = "."                 # folder with NB02_fPINN_*.npz (closed loop of reused rows is in NB03)
NB03_CSV     = "NB03_fPINN_ClosedLoop.csv"
CACHE_DIR    = "NB04_fPINN_cache"
WEIGHTS_OUT  = "NB04_fPINN_weights"
SPEED_SCALES, PAYLOADS = (6.4, 3.2, 1.6), (0.0, 6.0)

K4 = ("PINN", "HYBRID", "fPINN", "fHYBRID"); K5 = ("MLP", "HYBRID", "PINN", "fPINN", "fHYBRID")
def block(scn, alpha, kinds, sizes, seeds): return [(scn, alpha, k, n, s) for n in sizes for s in seeds for k in kinds]
if not QUICK:
    N_TRAIN_MAX = 175
    EXPERIMENTS = (block("null", 0.0, K4, (175,), (0, 1, 2))
                   + block("hist", ALPHA_MAIN, ("MLPhist", "PINNmem"), (175, 35), range(5))
                   + block("main", ALPHA_MAIN, K5, (175, 35), range(5))
                   + block("main", ALPHA_MAIN, ("PINN", "fPINN"), (175,), range(5, 10))
                   + block("bv0", ALPHA_MAIN, K4, (175,), (0, 1, 2))
                   + block("noise", ALPHA_MAIN, K4, (175,), (0, 1, 2))
                   + sum([block("sweep", a, K4, (175,), (0, 1, 2)) for a in (0.3, 0.7, 0.9)], []))
else:
    N_TRAIN_MAX = 35; SPEED_SCALES, PAYLOADS = (3.2,), (0.0,)
    TRAIN_CFG = dict(TRAIN_CFG, max_epochs=2, log_every=1)
    EXPERIMENTS = [("null", 0.0, "fPINN", 35, 0), ("hist", ALPHA_MAIN, "MLPhist", 35, 0), ("hist", ALPHA_MAIN, "PINNmem", 35, 0),
                   ("main", ALPHA_MAIN, "fPINN", 35, 0), ("bv0", ALPHA_MAIN, "fHYBRID", 35, 0), ("noise", ALPHA_MAIN, "PINN", 35, 0),
                   ("noise", ALPHA_MAIN, "fPINN", 35, 0), ("sweep", 0.3, "fPINN", 35, 1)]
E = TRAIN_CFG["max_epochs"]
ALPHAS_ALL = sorted({a for _, a, *_ in EXPERIMENTS if a > 0} | {ALPHA_MAIN})
# -----------------------------------------------------------------------------------------------
FS = 2000; dt = 1.0/FS; SUBSAMPLE = 5
DATA_SEED, N_VAL_TRAJ, N_TEST_TRAJ = 12345, 20, 20
ALPHA_GRID = np.round(np.arange(0.05, 1.0001, 0.025), 3); NA = len(ALPHA_GRID)
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(WEIGHTS_OUT, exist_ok=True)
torch.set_num_threads(min(8, max(1, (os.cpu_count() or 2)//2)))
print("QUICK" if QUICK else "FULL", f"| {len(EXPERIMENTS)} networks | epochs {E} | torch {torch.__version__} | threads {torch.get_num_threads()}")

## Robot model and unmodeled dynamics (NB01–NB03, notebook 18)

In [ ]:
R_BASE, R_PLAT, L_A, L_B = 0.114, 0.035, 0.200, 0.460
THETA0 = np.array([0.070, 0.023, 1e-5, 0.006, 0.253, 1.12, 0.008, 0.006, 0.006, 1.0, 0.008, 0.006, 0.006, 1.0])
M_BASE = 6.0
UNMODELED = dict(b_v=0.05, f_c=0.04, eps=0.02, c_cor=0.06, k_pos=0.04, w_pos=3.0)
def gamma_payload(m): return (M_BASE + np.asarray(m, float)) / M_BASE

def ik_vec(P):
    x, y, z = P[:, 0], P[:, 1], P[:, 2]; q = np.zeros((len(P), 3)); rd = R_BASE - R_PLAT
    for i in range(3):
        al = 2*i*np.pi/3; u = -2*L_A*z; aux = x*np.cos(al) + y*np.sin(al)
        v = 2*L_A*(rd - aux); w = L_B**2 - (rd**2 - 2*rd*aux + x**2 + y**2 + z**2 + L_A**2)
        r = np.sqrt(u**2 + v**2 - w**2); den = u**2 + v**2
        s1 = (u*w + v*r)/den; c1 = (v*w - u*r)/den; t1 = np.where(c1 >= 0, np.arcsin(s1), np.pi - np.arcsin(s1))
        s2 = (u*w - v*r)/den; c2 = (v*w + u*r)/den; t2 = np.where(c2 >= 0, np.arcsin(s2), np.pi - np.arcsin(s2))
        q[:, i] = np.where(np.abs(t1) < np.abs(t2), t1, t2)
    return q

def fk_vec(q):
    """Forward kinematics of NB03 (fk), vectorized over rows of q (n, 3)."""
    rd = R_BASE - R_PLAT; C = np.zeros((len(q), 3, 3))
    for i in range(3):
        a = 2*i*np.pi/3; r = rd + L_A*np.cos(q[:, i])
        C[:, i] = np.column_stack([r*np.cos(a), r*np.sin(a), L_A*np.sin(q[:, i])])
    A1, B1, C1 = (2*(C[:, 1] - C[:, 0])).T; D1 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 1]))
    A2, B2, C2 = (2*(C[:, 2] - C[:, 0])).T; D2 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 2]))
    den = A2*B1 - A1*B2
    L1 = (A1*D2 - A2*D1)/den; L2 = (A1*C2 - A2*C1)/den; L3 = (B2*D1 - B1*D2)/den; L4 = (B2*C1 - B1*C2)/den
    u = 1 + L2**2 + L4**2; c0 = C[:, 0]
    v = 2*(L3*L4 + L1*L2 - c0[:, 0]*L4 - c0[:, 1]*L2 - c0[:, 2])
    w = c0[:, 0]**2 + c0[:, 1]**2 + c0[:, 2]**2 + L3**2 + L1**2 - 2*c0[:, 0]*L3 - 2*c0[:, 1]*L1 - L_B**2
    z = (-v + np.sqrt(v**2 - 4*u*w)) / (2*u)
    return np.column_stack([L3 + L4*z, L1 + L2*z, z])
def fk(q): return fk_vec(q[None, :])[0]

def regressor_batch(q, a, P):
    n = len(q); s, c = np.sin(q), np.cos(q); x, y, z = P[:, 0], P[:, 1], P[:, 2]
    Y = np.zeros((n, 3, 14)); Y[:, :, 0] = a; Y[:, :, 4] = c
    for i, (cx, cy, cz, cs) in enumerate([(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]):
        Y[:, i, cx] = -x*s[:, i]; Y[:, i, cy] = -y*s[:, i]; Y[:, i, cz] = z*c[:, i]; Y[:, i, cs] = s[:, i]
    return Y
def regressor(q, a, p): return regressor_batch(q[None, :], np.asarray(a, float)[None, :], np.asarray(p)[None, :])[0]

def tau_unmodeled_batch(q, qd, sev=1.0, b_v=None):
    u = UNMODELED; bv = u["b_v"] if b_v is None else b_v
    return sev*(bv*qd + u["f_c"]*np.tanh(qd/u["eps"])
                + u["c_cor"]*(qd**2*np.cos(q) + np.roll(qd, -1, axis=1)*np.roll(qd, -2, axis=1))
                + u["k_pos"]*np.sin(u["w_pos"]*q))

def caputo_l1(x, h, alpha):
    x = np.asarray(x, float); N = len(x); k = np.arange(N); b = (k + 1.0)**(1 - alpha) - k**(1.0 - alpha)
    d = np.vstack([np.zeros((1, x.shape[1])), np.diff(x, axis=0)])
    return fftconvolve(d, b[:, None], axes=0)[:N] * h**(-alpha) / Gamma(2 - alpha)

def encoder(q, rng):
    """Quantized encoder reading (77 µrad) with Gaussian jitter of ENC_JITTER counts before quantization."""
    return np.round((q + ENC_JITTER*Q_ENC*rng.standard_normal(q.shape)) / Q_ENC) * Q_ENC

## Data: notebook 18 trajectories (same RNG), clean and encoder-measured versions

Clean data are those of NB02. In the measured version the inputs (q, q̇, q̈, p) and the logged history behind D^α q come from the quantized encoder (q̇, q̈ by Savitzky–Golay, p by forward kinematics), while the label is the torque the plant really received. Labels for the null test (B = 0) and for b_v = 0 are formed from stored components.

In [ ]:
def random_trajectory_full(rng, n_wp=6, v_range=(0.03, 0.8), t_min=0.3):     # RNG use identical to nb 18
    W = np.column_stack([rng.uniform(-0.12, 0.12, n_wp), rng.uniform(-0.12, 0.12, n_wp), rng.uniform(0.49, 0.56, n_wp)])
    dist = np.linalg.norm(np.diff(W, axis=0), axis=1)
    v = np.exp(rng.uniform(np.log(v_range[0]), np.log(v_range[1]), n_wp - 1))
    dur = np.maximum(dist / v, t_min); m_load = float(rng.integers(0, 7))
    edges = np.concatenate([[0.0], np.cumsum(dur)]); t_g = np.arange(0.0, edges[-1], dt)
    seg = np.clip(np.searchsorted(edges, t_g, side="right") - 1, 0, n_wp - 2)
    ts = (t_g - edges[seg]) / dur[seg]; s = 6*ts**5 - 15*ts**4 + 10*ts**3
    P = W[seg] + (W[seg + 1] - W[seg])*s[:, None]
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(q=q, qd=qd, qdd=qdd, P=P, m=m_load)

def log_and_label(T, margin_s=0.05, rng_meas=None):
    """NB02 sampling. Stores nominal torque (label), τ_u, its viscous part, the true fractional torques and the
    D^α grid features. With rng_meas, inputs, base and features come from the encoder; labels stay true."""
    q, qd, qdd, P, m = T["q"], T["qd"], T["qdd"], T["P"], T["m"]
    nom_y = gamma_payload(m)*np.einsum("nij,j->ni", regressor_batch(q, qdd, P), THETA0)
    frac = {a: B_FRAC*caputo_l1(q, dt, a) for a in ALPHAS_ALL}
    tu = tau_unmodeled_batch(q, qd); tv = UNMODELED["b_v"]*qd
    if rng_meas is not None:
        qm = encoder(q, rng_meas); w = min(SG_WINDOW, len(q) - (1 - len(q) % 2))
        qdm = savgol_filter(qm, w, 3, deriv=1, delta=dt, axis=0); qddm = savgol_filter(qm, w, 3, deriv=2, delta=dt, axis=0)
        Pm = fk_vec(qm); base = gamma_payload(m)*np.einsum("nij,j->ni", regressor_batch(qm, qddm, Pm), THETA0)
    else: qm, qdm, qddm, Pm, base = q, qd, qdd, P, nom_y
    mg = int(margin_s/dt); idx = np.arange(0, len(q), SUBSAMPLE); keep = (idx >= mg) & (idx < len(q) - mg)
    h = dt*SUBSAMPLE; ql = qm[idx]
    G = np.stack([caputo_l1(ql, h, a)[keep] for a in ALPHA_GRID], axis=1).astype(np.float32)   # (n, NA, 3)
    i = idx[keep]
    return dict(X=np.column_stack([qm[i], qdm[i], qddm[i], np.full(len(i), m), Pm[i]]), nom=base[i], nom_y=nom_y[i],
                tu=tu[i], tv=tv[i], G=G, **{f"frac_{a}": f[i] for a, f in frac.items()})

def build(n_traj, rng, rng_meas=None):
    parts = [log_and_label(random_trajectory_full(rng), rng_meas=rng_meas) for _ in range(n_traj)]
    D = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
    D["traj"] = np.concatenate([np.full(len(p["X"]), j) for j, p in enumerate(parts)]); return D

def build_all(noisy):
    rng = np.random.default_rng(DATA_SEED); rm = np.random.default_rng(777) if noisy else None
    return build(N_TRAIN_MAX, rng, rm), build(N_VAL_TRAJ, rng, rm), build(N_TEST_TRAJ, rng, rm)

t0 = time.time()
DATA = {"clean": build_all(False)}
if any(s == "noise" for s, *_ in EXPERIMENTS): DATA["noise"] = build_all(True)
print(f"data in {time.time()-t0:.0f} s | train {len(DATA['clean'][0]['X']):,} samples per set")
if "noise" in DATA:
    Dc, Dn = DATA["clean"][0], DATA["noise"][0]
    print("encoder data: rms error q %.1f µrad | q̇ %.4f rad/s | q̈ %.3f rad/s² (rms q̈ %.2f)" % (
        1e6*np.sqrt(np.mean((Dn['X'][:, :3]-Dc['X'][:, :3])**2)), np.sqrt(np.mean((Dn['X'][:, 3:6]-Dc['X'][:, 3:6])**2)),
        np.sqrt(np.mean((Dn['X'][:, 6:9]-Dc['X'][:, 6:9])**2)), np.sqrt(np.mean(Dc['X'][:, 6:9]**2))))
    del Dc, Dn

def labels(D, scn, alpha):
    """Label for each block: null = no fractional term; bv0 = no viscous term; others = NB02 plant."""
    y = D["nom_y"] + D["tu"]
    if scn == "bv0": y = y - D["tv"]
    if scn != "null": y = y + D[f"frac_{alpha}"]
    return dict(X=D["X"], y=y, base=D["nom"], G=D["G"], traj=D["traj"])
def subset(D, size):
    if size >= int(D["traj"].max()) + 1: return D
    keep = D["traj"] < size; return {k: v[keep] for k, v in D.items()}
def ram_gb():
    try:
        import psutil; return psutil.Process().memory_info().rss/1e9
    except Exception: return float("nan")

## Training: train_ff_model of NB02 + the two history baselines, with fast batches

`epoch_batches` returns the indices `DataLoader(shuffle=True, generator=torch.Generator().manual_seed(seed))` would produce: per epoch one draw for the worker seed, the permutation, and the extra permutation the sampler makes when it is exhausted.

In [ ]:
GA = torch.tensor(ALPHA_GRID, dtype=torch.float32); DA = float(ALPHA_GRID[1] - ALPHA_GRID[0])
def interp_D(G, alpha):
    u = torch.clamp((alpha - GA[0]) / DA, 0.0, len(ALPHA_GRID) - 1.000001)
    i0 = torch.floor(u).long(); w = u - i0
    return (1 - w)*G[:, i0, :] + w*G[:, i0 + 1, :]

def epoch_batches(n, gen, bs):
    torch.empty((), dtype=torch.int64).random_(generator=gen)
    perm = torch.randperm(n, generator=gen); torch.randperm(n, generator=gen)
    return perm.split(bs)

def train_ff_model(kind, d_tr, d_va, seed, cfg=TRAIN_CFG, use_dataloader=False):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    x_mean, x_std = d_tr["X"].mean(0), d_tr["X"].std(0) + 1e-8
    y_mean, y_std = d_tr["y"].mean(0), d_tr["y"].std(0)
    r_std = (d_tr["y"] - d_tr["base"]).std(0) + 1e-8
    PHYS_IDX = [0, 1, 2, 6, 7, 8, 10, 11, 12]
    frac = kind in ("fPINN", "fHYBRID"); hist_in = kind == "MLPhist"; mem = kind == "PINNmem"
    core = {"fPINN": "PINN", "fHYBRID": "HYBRID", "MLPhist": "MLP", "PINNmem": "PINN"}.get(kind, kind)
    use_G = frac or hist_in or mem
    if hist_in:
        Gf = d_tr["G"].reshape(len(d_tr["G"]), -1)
        g_mean = Gf.mean(0, dtype=np.float64); g_std = Gf.std(0, dtype=np.float64) + 1e-8
    else: g_mean = g_std = np.zeros(1)
    n_in = 13 + (3*NA if hist_in else 0)

    class FFNet(nn.Module):
        def __init__(self):
            super().__init__()
            n_out = 14 if core == "PINN" else 3
            self.net = nn.Sequential(nn.Linear(n_in, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, n_out))
            for name_, arr in [("y_mean", y_mean), ("y_std", y_std), ("r_std", r_std)]:
                self.register_buffer(name_, torch.tensor(arr, dtype=torch.float32))
            if core == "PINN":
                with torch.no_grad():
                    self.net[-1].weight.mul_(0.1); self.net[-1].bias.copy_(torch.tensor(THETA0, dtype=torch.float32))
            if frac:
                a0 = (ALPHA_INIT - 0.05)/0.925
                self.a_raw = nn.Parameter(torch.tensor(np.log(a0/(1 - a0)), dtype=torch.float32))
                self.B = nn.Parameter(torch.zeros(3))
            if mem: self.Wm = nn.Parameter(torch.zeros(NA, 3))
        def alpha(self): return 0.05 + 0.925*torch.sigmoid(self.a_raw)
        def forward(self, xs, xp, base, G):
            out = self.net(xs)
            if core == "MLP": tau = out*self.y_std + self.y_mean
            elif core == "HYBRID": tau = base + out*self.r_std
            else:
                q, qdd = xp[:, 0:3], xp[:, 3:6]; x, y, z = xp[:, 6], xp[:, 7], xp[:, 8]
                th = [out[:, i] for i in range(14)]
                t1 = (th[0]*qdd[:, 0] - th[1]*x*torch.sin(q[:, 0]) - th[2]*y*torch.sin(q[:, 0])
                      + th[3]*z*torch.cos(q[:, 0]) + th[4]*torch.cos(q[:, 0]) + th[5]*torch.sin(q[:, 0]))
                t2 = (th[0]*qdd[:, 1] - th[6]*x*torch.sin(q[:, 1]) - th[7]*y*torch.sin(q[:, 1])
                      + th[8]*z*torch.cos(q[:, 1]) + th[4]*torch.cos(q[:, 1]) + th[9]*torch.sin(q[:, 1]))
                t3 = (th[0]*qdd[:, 2] - th[10]*x*torch.sin(q[:, 2]) - th[11]*y*torch.sin(q[:, 2])
                      + th[12]*z*torch.cos(q[:, 2]) + th[4]*torch.cos(q[:, 2]) + th[13]*torch.sin(q[:, 2]))
                tau = torch.stack([t1, t2, t3], dim=1)
            if frac: tau = tau + self.B*interp_D(G, self.alpha())
            if mem: tau = tau + (G*self.Wm).sum(1)
            return tau
        def memory_torque(self, G):
            if frac: return self.B*interp_D(G, self.alpha())
            if mem: return (G*self.Wm).sum(1)
            return torch.zeros(len(G), 3)

    def to_tensors(d):
        xs = (d["X"] - x_mean)/x_std
        if hist_in: xs = np.hstack([xs, (d["G"].reshape(len(d["G"]), -1) - g_mean)/g_std])
        xs = torch.tensor(xs, dtype=torch.float32)
        xp = torch.tensor(d["X"][:, PHYS_IDX], dtype=torch.float32)
        G = torch.from_numpy(d["G"]) if use_G else torch.zeros(len(xs), 1, 1)
        return TensorDataset(xs, xp, torch.tensor(d["base"], dtype=torch.float32), G, torch.tensor(d["y"], dtype=torch.float32))

    ds_tr, ds_va = to_tensors(d_tr), to_tensors(d_va)
    if use_dataloader:
        train_loader = DataLoader(ds_tr, batch_size=cfg["batch_size"], shuffle=True, generator=torch.Generator().manual_seed(seed))
        get_batches = lambda: train_loader
    else:
        gen = torch.Generator().manual_seed(seed); T_tr = ds_tr.tensors; n_tr = len(T_tr[0])
        get_batches = lambda: ([t[idx] for t in T_tr] for idx in epoch_batches(n_tr, gen, cfg["batch_size"]))
    T_va = ds_va.tensors
    model = FFNet()
    extra = ([model.a_raw, model.B] if frac else []) + ([model.Wm] if mem else [])
    groups = [{"params": model.net.parameters(), "weight_decay": cfg["weight_decay"]}]
    if extra: groups.append({"params": extra, "weight_decay": 0.0})
    opt = torch.optim.Adam(groups, lr=cfg["lr"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=cfg["lr_patience"])
    loss_fn = nn.MSELoss(); best, best_state, wait = np.inf, None, 0; t_tr = time.time(); losses = []
    for epoch in range(1, cfg["max_epochs"] + 1):
        model.train(); ls = 0.0
        for xs, xp, base, G, y in get_batches():
            opt.zero_grad(); loss = loss_fn(model(xs, xp, base, G), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); ls += loss.item()
        losses.append(ls)
        model.eval(); vs, vn = 0.0, 0
        with torch.no_grad():
            for k0 in range(0, len(T_va[0]), cfg["batch_size"]):
                xs, xp, base, G, y = [t[k0:k0 + cfg["batch_size"]] for t in T_va]
                vs += loss_fn(model(xs, xp, base, G), y).item()*len(y); vn += len(y)
        val = vs/vn; sched.step(val)
        if val < best - 1e-12: best, best_state, wait = val, copy.deepcopy(model.state_dict()), 0
        else: wait += 1
        if epoch % cfg["log_every"] == 0 or epoch == 1 or wait >= cfg["patience"]:
            ex = f" | α {model.alpha().item():.3f} B {np.round(model.B.detach().numpy(), 4)}" if frac else ""
            print(f"    [{kind} {d_tr['traj'].max()+1}traj seed{seed}] epoch {epoch:3d} | val MSE {val:.3e} | best {best:.3e}{ex} | {time.time()-t_tr:.0f} s", flush=True)
        if wait >= cfg["patience"]: break
    del ds_tr, ds_va, opt, sched, get_batches
    model.load_state_dict(best_state); model.eval()
    info = dict(epochs=epoch, best_val_mse=best, train_s=time.time() - t_tr, losses=losses)
    lins = [m_ for m_ in model.net if isinstance(m_, nn.Linear)]
    info["export"] = dict(kind=kind, **{k: t_.detach().numpy().astype(float) for k, t_ in
                          zip(("W1", "b1", "W2", "b2", "W3", "b3"), [t for l_ in lins for t in (l_.weight, l_.bias)])},
                          x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std, r_std=r_std, g_mean=g_mean, g_std=g_std,
                          alpha=model.alpha().item() if frac else 0.0, B=model.B.detach().numpy().astype(float) if frac else np.zeros(3),
                          Wm=model.Wm.detach().numpy().astype(float) if mem else np.zeros((NA, 3)), alpha_grid=ALPHA_GRID)
    if frac: info.update(alpha_hat=model.alpha().item(), B_hat=model.B.detach().numpy().astype(float))
    info["n_weights"] = sum(p.numel() for p in model.parameters())
    return model, to_tensors, info

def evaluate(model, to_tensors, d):
    xs, xp, base, G, y = to_tensors(d).tensors
    with torch.no_grad():
        err = torch.sqrt(torch.mean((model(xs, xp, base, G) - y)**2)).item()
        mt = torch.sqrt(torch.mean(model.memory_torque(G)**2)).item()
    return err, mt

## Check: fast batches reproduce DataLoader (one epoch, PINN, 35 trajectories)

In [ ]:
if CHECK_LOADER:
    cfg1 = dict(TRAIN_CFG, max_epochs=1, log_every=1000)
    d35 = subset(DATA["clean"][0], 35); tr = labels(d35, "main", ALPHA_MAIN); va = labels(DATA["clean"][1], "main", ALPHA_MAIN)
    t_a = time.time(); _, _, ia = train_ff_model("fPINN", tr, va, 0, cfg1, use_dataloader=True); t_a = time.time() - t_a
    t_b = time.time(); _, _, ib = train_ff_model("fPINN", tr, va, 0, cfg1, use_dataloader=False); t_b = time.time() - t_b
    same = abs(ia["losses"][0] - ib["losses"][0]) <= 1e-6*abs(ia["losses"][0])
    print(f"epoch loss DataLoader {ia['losses'][0]:.8e} | fast {ib['losses'][0]:.8e} | {'IDENTICAL' if same else 'DIFFERENT (statistically equivalent; reused NB02 seeds are still valid)'}"
          f" | {t_a:.1f} s vs {t_b:.1f} s per epoch")
    del d35, tr, va, ia, ib; gc.collect()

## Adept references for the open-loop torque check (clean and encoder-measured)

In [ ]:
P_READY = np.array([0.0, 0.0, 0.5])
WAYPOINTS = [P_READY, np.array([.10, .10, .55]), np.array([.10, -.10, .55]),
             np.array([-.10, -.10, .55]), np.array([-.10, .10, .55]), np.array([.10, .10, .55]), P_READY]
BASE_T = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.50]); H_P = 0.05
ALL_SCALES = (6.4, 3.2, 1.6)
def adept_full(scale, m=0.0):
    nt = BASE_T*scale; n = int(round(nt[-1]*FS)) + 1; t = np.arange(n)*dt
    seg = np.clip(np.searchsorted(nt[1:], t, side="right"), 0, len(WAYPOINTS) - 2)
    tau = np.clip((t - nt[seg]) / (nt[seg+1] - nt[seg]), 0, 1); s = 6*tau**5 - 15*tau**4 + 10*tau**3
    W = np.array(WAYPOINTS); P = W[seg] + (W[seg+1] - W[seg])*s[:, None]; P[:, 2] -= H_P*np.sin(np.pi*tau)**2
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(t=t, q=q, qd=qd, qdd=qdd, P=P, m=m, N=n)
REFS_OL = {}
for nz in DATA:
    for sc in ALL_SCALES:
        for m in (0.0, 6.0):
            R = log_and_label(adept_full(sc, m), margin_s=0.0, rng_meas=np.random.default_rng(99) if nz == "noise" else None)
            R["traj"] = np.zeros(len(R["X"]), int); REFS_OL[(nz, sc, m)] = R

## Run every network (blocks in priority order; NB02 rows reused)

In [ ]:
LOG_CSV = "NB04_fPINN_TrainingLog" + ("_QUICK" if QUICK else "") + ".csv"   # QUICK writes apart, so it never mixes with the full run
def tag(scn, alpha, kind, size, seed): return f"{scn}_{kind}_{size}traj_seed{seed}_a{alpha}_e{E}"
rows, sec_epoch = [], {}; t_all = time.time()
def eta(j):
    """Remaining training time from the measured s/epoch (median epochs 110 at 175 traj, 150 at 35)."""
    done_ = lambda sc_, a_, k_, n_, sd_: (os.path.exists(os.path.join(CACHE_DIR, tag(sc_, a_, k_, n_, sd_) + ".pkl")) or
             (sc_ in ("main", "sweep") and os.path.exists(os.path.join(NB02_CACHE, f"{k_}_{n_}traj_seed{sd_}_a{a_}_e{E}.pkl"))))
    left = [x for x in EXPERIMENTS[j:] if not done_(*x)]; s175 = np.median(sec_epoch.get(175, [1.4])); s35 = np.median(sec_epoch.get(35, [s175/4]))
    return sum((s175*110 if x[3] == 175 else s35*150) for x in left)
print(f"estimated training time on this PC before measuring: {eta(0)/3600:.1f} h (+ closed loop, see below)")
for j, (scn, alpha, kind, size, seed) in enumerate(EXPERIMENTS, 1):
    name = tag(scn, alpha, kind, size, seed); f_c = os.path.join(CACHE_DIR, name + ".pkl"); f_w = os.path.join(WEIGHTS_OUT, f"NB04_fPINN_{name}.npz")
    if os.path.exists(f_c) and os.path.exists(f_w):
        rows.append(pickle.load(open(f_c, "rb"))); print(f"[{j}/{len(EXPERIMENTS)}] {name}: loaded from cache"); continue
    f02 = os.path.join(NB02_CACHE, f"{kind}_{size}traj_seed{seed}_a{alpha}_e{E}.pkl")
    if scn in ("main", "sweep") and os.path.exists(f02):
        row, _ = pickle.load(open(f02, "rb")); row = dict(row, Scenario=scn, Source="NB02")
        rows.append(row); print(f"[{j}/{len(EXPERIMENTS)}] {name}: reused from NB02"); continue
    nz = "noise" if scn == "noise" else "clean"; Dtr, Dva, Dte = DATA[nz]
    TR, VA, TE = labels(subset(Dtr, size), scn, alpha), labels(Dva, scn, alpha), labels(Dte, scn, alpha)
    model, tt, info = train_ff_model(kind, TR, VA, seed)
    te, te_mem = evaluate(model, tt, TE)
    row = dict(Scenario=scn, Source="NB04", **{"alpha true": alpha}, Kind=kind, Trajectories=size, Seed=seed, Epochs=info["epochs"],
               **{"Train time (s)": info["train_s"], "Weights": info["n_weights"], "Val RMSE (N m)": evaluate(model, tt, VA)[0],
                  "Test RMSE (N m)": te, "Memory torque RMS (N m)": te_mem})
    for sc in ALL_SCALES:
        for m in (0.0, 6.0): row[f"Adept {sc} {m:.0f} kg (N m)"] = evaluate(model, tt, labels(REFS_OL[(nz, sc, m)], scn, alpha))[0]
    if "alpha_hat" in info:
        row["alpha hat"] = info["alpha_hat"]; row["B hat mean"] = float(info["B_hat"].mean()); row["B hat"] = [round(float(b), 4) for b in info["B_hat"]]
    np.savez(f_w, **info["export"]); pickle.dump(row, open(f_c, "wb"))
    rows.append(row); pd.DataFrame(rows).to_csv(LOG_CSV, index=False)
    sec_epoch.setdefault(size, []).append(info["train_s"]/info["epochs"])
    print(f"[{j}/{len(EXPERIMENTS)}] {name}: test {te*1e3:.2f} mN m" + (f" | α̂ {row['alpha hat']:.3f} B̂ {row['B hat']}" if "alpha hat" in row else "")
          + f" | memory torque {te_mem*1e3:.2f} mN m | {info['train_s']/info['epochs']:.2f} s/epoch | {(time.time()-t_all)/60:.0f} min"
          + f" | training left ≈ {eta(j)/3600:.1f} h | RAM {ram_gb():.2f} GB", flush=True)
    del model, tt, info, TR, VA, TE; gc.collect()
df = pd.DataFrame(rows); df.to_csv(LOG_CSV, index=False)
del DATA; _ = gc.collect()

## Closed loop (CTC + I of NB03) for every new network of main, sweep, hist and noise

Same plant, gains, torque limit (5 N m) and memory update (2 ms) as NB03. `PINNmem` adds Σ_j w_j D^αj q from the measured history; `MLPhist` receives the 39 features as inputs (held between 2 ms updates). In `noise` the controller reads the quantized encoder (fresh jitter every step) through an α-β observer of bandwidth `OBS_BW` (integrated like the plant and predicted with the reference acceleration, so it adds no bias without noise; checked here: oracle 10 µm vs 9.2 µm clean at 3.2 s), and computes p by forward kinematics of the estimated angles; the nominal model and the oracle run under the same measurement.

In [ ]:
KP_PD, KD_PD, KI_RATIO = 150.0, 10.0, 10.0
KP_EQ, KD_EQ = KP_PD/THETA0[0], KD_PD/THETA0[0]
T_C = 0.002; TAU_MAX = 5.0

def silu(z): return z / (1.0 + np.exp(-z))
class NumpyFF:
    def __init__(self, path):
        d = np.load(path, allow_pickle=False); self.kind = str(d["kind"])
        self.core = {"fPINN": "PINN", "fHYBRID": "HYBRID", "MLPhist": "MLP", "PINNmem": "PINN"}.get(self.kind, self.kind)
        self.W = [d[k].astype(float) for k in ("W1", "b1", "W2", "b2", "W3", "b3")]
        self.x_mean, self.x_std = d["x_mean"].astype(float), d["x_std"].astype(float)
        self.y_mean, self.y_std, self.r_std = d["y_mean"].astype(float), d["y_std"].astype(float), d["r_std"].astype(float)
        self.alpha, self.B = float(d["alpha"]), d["B"].astype(float)
        self.g_mean = d["g_mean"].astype(float) if "g_mean" in d else np.zeros(1); self.g_std = d["g_std"].astype(float) if "g_std" in d else np.ones(1)
        self.Wm = d["Wm"].astype(float) if "Wm" in d else np.zeros((NA, 3)); self.feat = np.zeros(3*NA)
    def F(self, q, qd, qdd, m):
        p = fk(q); xs = (np.concatenate([q, qd, qdd, [m], p]) - self.x_mean) / self.x_std
        if self.kind == "MLPhist": xs = np.concatenate([xs, (self.feat - self.g_mean)/self.g_std])
        W1, b1, W2, b2, W3, b3 = self.W; out = W3 @ silu(W2 @ silu(W1 @ xs + b1) + b2) + b3
        if self.core == "MLP": return out*self.y_std + self.y_mean
        if self.core == "HYBRID": return gamma_payload(m)*(regressor(q, qdd, p) @ THETA0) + out*self.r_std
        return regressor(q, qdd, p) @ out

class FracHistory:
    """Online L1 Caputo derivative (full memory) for one or several orders; returns (n_orders, 3)."""
    def __init__(self, alphas, h, n_max):
        a = np.atleast_1d(np.asarray(alphas, float))[:, None]; k = np.arange(n_max + 1)[None, :]
        self.b = ((k + 1.0)**(1 - a) - k**(1.0 - a))[:, ::-1].copy(); self.c = (h**(-a[:, 0]) / np.array([Gamma(2 - x) for x in a[:, 0]]))[:, None]
        self.dq = np.zeros((n_max + 1, 3)); self.n = 0; self.q_last = None
    def push(self, q):
        if self.q_last is None: self.q_last = q.copy(); return np.zeros((len(self.c), 3))
        self.n += 1; self.dq[self.n] = q - self.q_last; self.q_last = q.copy()
        return self.c * (self.b[:, -self.n:] @ self.dq[1:self.n + 1])

def F_nominal(q, qd, qdd, m): return gamma_payload(m) * (regressor(q, qdd, fk(q)) @ THETA0)
def F_oracle(q, qd, qdd, m): return F_nominal(q, qd, qdd, m) + tau_unmodeled_batch(q[None], qd[None])[0]

class Controller:
    """CTC + I (NB03) + memory term updated every T_C from the measured angles."""
    def __init__(self, F, mode="none", alpha=0.5, B=np.zeros(3), net=None):
        self.F, self.mode, self.alpha, self.B, self.net = F, mode, alpha, np.asarray(B, float), net; self.step = int(round(T_C/dt))
    def reset(self, N):
        self.I = np.zeros(3); self.mem = np.zeros(3); n_max = N // self.step + 2
        self.H = (FracHistory(self.alpha, T_C, n_max) if self.mode == "frac" else
                  FracHistory(ALPHA_GRID, T_C, n_max) if self.mode in ("mem", "hist") else None)
    def __call__(self, R, i, q, qd, m):
        if self.H is not None and i % self.step == 0:
            D = self.H.push(q)
            if self.mode == "frac": self.mem = self.B*D[0]
            elif self.mode == "mem": self.mem = (self.net.Wm*D).sum(0)
            else: self.net.feat = D.reshape(-1)
        e = R["q"][i] - q; ed = R["qd"][i] - qd; self.I += e*dt
        v = R["qdd"][i] + KD_EQ*ed + KP_EQ*e + KI_RATIO*KP_EQ*self.I
        return self.F(q, qd, v, m) + self.mem

def plant_qdd(q, qd, tau, p, m, Dq):
    g = gamma_payload(m)
    return (tau - g*(regressor(q, np.zeros(3), p) @ THETA0) - tau_unmodeled_batch(q[None], qd[None])[0] - B_FRAC*Dq) / (g*THETA0[0])

def simulate(ctrl, R, m, alpha, noisy, seed=0):
    """NB03 loop. With noisy = True the controller sees an α-β observer (bandwidth OBS_BW) on the quantized encoder,
    integrated like the plant and predicted with the reference acceleration, so it has no bias without noise."""
    ctrl.reset(R["N"]); q = R["q"][0].copy(); qd = R["qd"][0].copy(); Hp = FracHistory(alpha, dt, R["N"])
    rng = np.random.default_rng(seed); w = 2*np.pi*OBS_BW; l1, l2 = 2*w*dt, w*w*dt
    qhp, qdhp = q.copy(), qd.copy()
    P = np.zeros((R["N"], 3)); T = np.zeros((R["N"], 3))
    for i in range(R["N"]):
        p = fk(q); P[i] = p; Dq = Hp.push(q)[0]
        if noisy:
            r = encoder(q, rng) - qhp; qh = qhp + l1*r; qdh = qdhp + l2*r
            tau = ctrl(R, i, qh, qdh, m); qdhp = qdh + R["qdd"][i]*dt; qhp = qh + qdhp*dt
        else: tau = ctrl(R, i, q, qd, m)
        tau = np.clip(tau, -TAU_MAX, TAU_MAX); T[i] = tau
        qd = qd + plant_qdd(q, qd, tau, p, m, Dq)*dt; q = q + qd*dt
        if not np.all(np.isfinite(q)) or np.abs(q).max() > 3: return None
    e = 1000*np.linalg.norm(R["P"] - P, axis=1)
    return {"Cartesian RMSE (mm)": np.sqrt(np.mean(e**2)), "Max error (mm)": e.max(), "RMS torque (N m)": np.sqrt(np.mean(T**2)),
            "TV (N m)": np.sum(np.linalg.norm(np.diff(T, axis=0), axis=1)), "Peak |tau| (N m)": np.abs(T).max()}

REFS_CL = {sc: adept_full(sc) for sc in SPEED_SCALES}
def models_for_cl():
    """(scenario, alpha, label, size, seed, controller factory) for every closed-loop model."""
    out = []
    for scn, alpha, kind, size, seed in EXPERIMENTS:
        if scn not in ("main", "sweep", "hist", "noise"): continue
        f = os.path.join(WEIGHTS_OUT, f"NB04_fPINN_{tag(scn, alpha, kind, size, seed)}.npz")
        if not os.path.exists(f): f = os.path.join(NB02_WEIGHTS, f"NB02_fPINN_{kind}_{size}traj_seed{seed}_a{alpha}_e{E}.npz")
        if not os.path.exists(f): print("missing weights:", tag(scn, alpha, kind, size, seed)); continue
        def make(f=f):
            n = NumpyFF(f); mode = {"MLPhist": "hist", "PINNmem": "mem"}.get(n.kind, "frac" if n.B.any() else "none")
            return Controller(n.F, mode, n.alpha, n.B, n), (n.alpha if n.B.any() else np.nan), n.B.mean()
        out.append((scn, alpha, kind, size, seed, make))
    for scn in ("main", "sweep", "noise"):                       # references (NB03 already holds main / sweep ones)
        for alpha in sorted({a for s, a, *_ in EXPERIMENTS if s == scn}):
            out.append((scn, alpha, "nominal", 0, 0, lambda: (Controller(F_nominal), np.nan, 0.0)))
            out.append((scn, alpha, "oracle (frac)", 0, 0, lambda a=alpha: (Controller(F_oracle, "frac", a, np.full(3, B_FRAC)), a, B_FRAC)))
    return out

CL_CSV = "NB04_fPINN_ClosedLoop" + ("_QUICK" if QUICK else "") + ".csv"
key = lambda d: (d["Scenario"], float(d["alpha true"]), d["Model"], int(d["Data (traj.)"]), int(d["Seed"]), float(d["Speed scale"]), float(d["Payload (kg)"]))
cl_rows = pd.read_csv(CL_CSV).to_dict("records") if os.path.exists(CL_CSV) else []
if not cl_rows and os.path.exists(NB03_CSV) and not QUICK:
    old = pd.read_csv(NB03_CSV); old.insert(0, "Scenario", np.where(old["alpha true"] == ALPHA_MAIN, "main", "sweep"))
    old.insert(1, "Source", "NB03"); cl_rows = old.to_dict("records"); print(f"{len(cl_rows)} closed-loop runs reused from NB03")
done = {key(r) for r in cl_rows}
CL_MODELS = models_for_cl(); jobs = [(M, sc, m) for M in CL_MODELS for sc in SPEED_SCALES for m in PAYLOADS]
todo = [(M, sc, m) for M, sc, m in jobs if key(dict(Scenario=M[0], **{"alpha true": M[1]}, Model=M[2], **{"Data (traj.)": M[3]}, Seed=M[4],
                                                  **{"Speed scale": sc, "Payload (kg)": m})) not in done]
print(f"closed loop: {len(todo)} runs to do (≈ {len(todo)*5/60:.0f} min)")
t_cl = time.time()
for j, ((scn, alpha, label, size, seed, make), sc, m) in enumerate(todo, 1):
    ctrl, a_hat, b_hat = make(); t0 = time.time()
    res = simulate(ctrl, REFS_CL[sc], m, alpha, noisy=(scn == "noise"), seed=1000*seed + int(10*sc) + int(m))
    base = dict(Scenario=scn, Source="NB04", **{"alpha true": alpha, "Controller": "CTC + I"}, Model=label, **{"Data (traj.)": size}, Seed=seed,
                **{"Speed scale": sc, "Payload (kg)": m, "alpha hat": a_hat, "B hat mean": b_hat, "Run time (s)": time.time() - t0})
    cl_rows.append(dict(base, Unstable=res is None, **(res or {})))
    pd.DataFrame(cl_rows).to_csv(CL_CSV, index=False)
    print(f"[{j}/{len(todo)}] {scn} α {alpha} | {sc} s {m:.0f} kg | {label:13s} {size:3d} traj s{seed} | "
          + ("UNSTABLE" if res is None else f"RMSE {1e3*res['Cartesian RMSE (mm)']:.1f} µm") + f" | {time.time()-t0:.0f} s | {(time.time()-t_cl)/60:.0f} min", flush=True)
cl = pd.DataFrame(cl_rows)

## Summary: the questions above, with paired statistics over seeds

In [ ]:
from scipy import stats
pd.set_option("display.width", 250); out = []
def paired(a, b, name):
    """a, b: Series indexed by seed. Mean difference, 95 % CI, paired t and Wilcoxon p-values."""
    s = a.index.intersection(b.index); d = (a[s] - b[s]).values
    if len(d) < 2: return dict(Comparison=name, n=len(d))
    ci = stats.t.ppf(0.975, len(d) - 1)*d.std(ddof=1)/np.sqrt(len(d))
    w = stats.wilcoxon(d).pvalue if len(d) >= 5 and np.any(d != 0) else np.nan
    return dict(Comparison=name, n=len(d), **{"mean diff": d.mean(), "CI95 half": ci, "p (paired t)": stats.ttest_rel(a[s], b[s]).pvalue, "p (Wilcoxon)": w})
def series(scn, kind, size, col, alpha=ALPHA_MAIN):
    g = df[(df.Scenario == scn) & (df.Kind == kind) & (df.Trajectories == size) & (df["alpha true"] == alpha)]
    return g.set_index("Seed")[col]
print("Torque RMSE (mN m), mean ± std over seeds:")
cols = ["Test RMSE (N m)", "Adept 1.6 0 kg (N m)", "Adept 1.6 6 kg (N m)"]
tab = df.groupby(["Scenario", "alpha true", "Kind", "Trajectories"])[cols].agg(["mean", "std", "count"])
print((tab*np.array([1e3, 1e3, 1]*3)).round(2).to_string())
if "alpha hat" in df:
    print("\nIdentified fractional parameters (mean ± std over seeds):")
    print(df.dropna(subset=["alpha hat"]).groupby(["Scenario", "alpha true", "Kind", "Trajectories"])[["alpha hat", "B hat mean", "Memory torque RMS (N m)"]].agg(["mean", "std"]).round(4).to_string())
for scn, size, alpha in [("main", 175, ALPHA_MAIN), ("main", 35, ALPHA_MAIN), ("noise", 175, ALPHA_MAIN), ("bv0", 175, ALPHA_MAIN), ("null", 175, 0.0)] + [("sweep", 175, a) for a in (0.3, 0.7, 0.9)]:
    for c in cols:
        out.append(dict(Scenario=scn, alpha=alpha, Trajectories=size, Metric=c, **paired(series(scn, "fPINN", size, c, alpha), series(scn, "PINN", size, c, alpha), "fPINN - PINN")))
for size in (175, 35):
    for k in ("MLPhist", "PINNmem"):
        for c in cols:
            out.append(dict(Scenario="hist", alpha=ALPHA_MAIN, Trajectories=size, Metric=c, **paired(series("main", "fPINN", size, c), series("hist", k, size, c), f"fPINN - {k}")))
if len(cl):
    ok = cl[~cl["Unstable"].astype(bool)]
    for scn in ("main", "noise"):
        for sc in SPEED_SCALES:
            for m in PAYLOADS:
                g = ok[(ok.Scenario == scn) & (ok["alpha true"] == ALPHA_MAIN) & (ok["Data (traj.)"] == 175) & (ok["Speed scale"] == sc) & (ok["Payload (kg)"] == m)]
                a = g[g.Model == "fPINN"].set_index("Seed")["Cartesian RMSE (mm)"]; b = g[g.Model == "PINN"].set_index("Seed")["Cartesian RMSE (mm)"]
                out.append(dict(Scenario=scn + " closed loop", alpha=ALPHA_MAIN, Trajectories=175, Metric=f"Cartesian RMSE {sc} s {m:.0f} kg (mm)", **paired(a, b, "fPINN - PINN")))
summ = pd.DataFrame(out); summ.to_csv("NB04_fPINN_Summary" + ("_QUICK" if QUICK else "") + ".csv", index=False)
print("\nPaired comparisons over seeds (negative mean diff = first model better):"); print(summ.round(5).to_string(index=False))

## Figure: test error per block (PINN vs fPINN and history baselines) and α̂ in the null / noise / b_v = 0 tests

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
groups = [("main", "PINN"), ("main", "fPINN"), ("hist", "PINNmem"), ("hist", "MLPhist"), ("noise", "PINN"), ("noise", "fPINN"), ("bv0", "PINN"), ("bv0", "fPINN"), ("null", "PINN"), ("null", "fPINN")]
data = [1e3*df[(df.Scenario == s) & (df.Kind == k) & (df.Trajectories == df.Trajectories.max())]["Test RMSE (N m)"].values for s, k in groups]
keep = [i for i, d in enumerate(data) if len(d)]
ax[0].boxplot([data[i] for i in keep]); ax[0].set_xticks(range(1, len(keep) + 1)); ax[0].set_xticklabels([f"{groups[i][0]}\n{groups[i][1]}" for i in keep], fontsize=8)
ax[0].set_ylabel("Test torque RMSE (mN m)"); ax[0].set_title(f"(a) {df.Trajectories.max()} trajectories, all seeds", loc="left")
fr = df.dropna(subset=["alpha hat"])
for i, (s, mk) in enumerate([("main", "o"), ("noise", "s"), ("bv0", "^"), ("null", "x"), ("sweep", "d")]):
    g = fr[fr.Scenario == s]
    if len(g): ax[1].scatter(g["alpha true"] + 0.01*i, g["alpha hat"], marker=mk, s=22, label=f"{s} ({'B̂≈%.3f' % g['B hat mean'].mean()})")
ax[1].plot([0, 1], [0, 1], "k:", lw=0.8); ax[1].set_xlabel("true α (0 = null test, B = 0)"); ax[1].set_ylabel("α̂"); ax[1].set_title("(b) identified order", loc="left"); ax[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig("fig_nb04_fpinn.png", dpi=200); fig.savefig("fig_nb04_fpinn.pdf"); plt.show()